# **LOS/NLOS Classifier Model**

## **Project Setup**

### **Import libraries**
Import all basic libraries here. Additional libraries needed later in the notebook can be done at the code cells they are needed for.

In [ ]:
# import libraries

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

### **Load data**
Load all 7 indoor environment datasets into a single dataframe

In [ ]:
dataset_path = "../dataset/"  # Dataset directory

csv_files = [
    f for f in os.listdir(dataset_path) if f.endswith(".csv")
]  # Retrieve all csv files in the dataset folder

print(csv_files)  # print out all the csv file name in the 'dataset' folder

# Load and print out dataset values
dfs = []
for file in csv_files:
    print(f"Reading file: {file}")  # Print the file name being read
    df_temp = pd.read_csv(os.path.join(dataset_path, file))  # Read the CSV file
    # print(df_temp)  # Print the entire data of the current CSV file
    dfs.append(df_temp)  # Append the DataFrame to the list

df = pd.concat(
    dfs, ignore_index=True
)  # combining all datasets (csv) files into one DataFrame
print(df)

Reading the shape and columns of the dataframe provides information on all the data.

In [ ]:
# 'df' variable => combine all dataframe into one
print(
    "Number of rows and Column in dataset:", df.shape
)  # shape of the datasets (the total columns and rolls -> matrix[columns,rows])
print(
    "Number of rows in dataset:", df.columns
)  # total columns/attribute in the all datasets

print(
    "number of records in overall datasets:", len(df)
)  # Print the number of records in the all datasets

## **1.1 -  Data Preparation (Data Cleaning & Data Preprocessing)**

### **1.1.1 - Data Cleaning**

For data cleaning, we counted the total number of missing (NaN) values in the dataframe. If there is existing missing values, the rows are dropped and the new dataframe is returned to df_clean.

1. Check and Handle Missing Values

In [ ]:
# Calculate the missing values for each column
MissingValCount = df.isnull().sum()

# If there are missing values, drop them and create a new cleaned dataset
if MissingValCount.sum() > 0:
    df_clean = df.dropna()  # Drop rows with missing values
    print(
        f"Total missing values found and removed: {MissingValCount.sum()} rows dropped."
    )
else:
    df_clean = df.copy()  # If no missing values, just copy the original dataset
    print("No missing values found in datasets")

2. Display Dataset Shape Before and After Cleaning

In [ ]:
# Display the shape of the dataset before and after cleaning
print("Before cleaning, dataset rows and columns:", df.shape)
print("After cleaning, dataset rows and columns:", df_clean.shape)

# Check if there is a difference in rows or columns
if df.shape == df_clean.shape:
    print("The number of rows and columns are the same in both datasets.")
else:
    rows_diff = df.shape[0] - df_clean.shape[0]
    cols_diff = df.shape[1] - df_clean.shape[1]
    print(
        "The number of rows and columns are different between the original and cleaned datasets.",
        f"Rows difference: {rows_diff}, Columns difference: {cols_diff}",
    )

### 1.1.2 - **Data Visualization and Data Cleaning**

Next, we will perform data visualization to better understand the features and their relationships. Visualizing the data helps in identifying patterns, trends, and potential outliers.

In [ ]:
# import libraries
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

#### **1.1.2.1 - Target Variable: NLOS**

**Count Occurrences of Values**

In [ ]:
# count the total in NLOS attribute in the datasets
count_NLOS = df_clean["NLOS"].value_counts()

print(count_NLOS)

**Visualizing the Data**

The pie chart below illustrates the count for **NLOS** with labels 0.0 and 1.0. From the graph, we can observe that the distribution is perfectly balanced, with an equal count of 0.0 and 1.0 labels.

In [ ]:
plt.figure(figsize=(6, 10))
count_NLOS.plot.pie(autopct="%1.1f%%", colors=["#3498db", "#e74c3c"], startangle=90)
plt.title("Distribution of NLOS")
plt.ylabel("")
plt.show()

#### **1.1.2.2 - Target Variable: FP_IDX**


**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

1. Calculate IQR, Identify and Filter Outliers

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) for FP_IDX
Q1 = df_clean["FP_IDX"].quantile(0.25)
Q3 = df_clean["FP_IDX"].quantile(0.75)

# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers in FP_IDX based on the bounds
outliers = df_clean[
    (df_clean["FP_IDX"] < lower_bound) | (df_clean["FP_IDX"] > upper_bound)
]

# Print outliers
print("Outliers for 'FP_IDX':\n", outliers)

# Filter out outliers separately for LOS and NLOS
# For LOS (Line of Sight)
df_clean_filtered_LOS = df_clean[
    (df_clean["FP_IDX"] >= lower_bound)
    & (df_clean["FP_IDX"] <= upper_bound)
    & (df_clean["NLOS"] == 0.0)  # LOS condition
]

# For NLOS (Non-Line of Sight)
df_clean_filtered_NLOS = df_clean[
    (df_clean["FP_IDX"] >= lower_bound)
    & (df_clean["FP_IDX"] <= upper_bound)
    & (df_clean["NLOS"] == 1.0)  # NLOS condition
]

2. Combine Data, Clean Redundant Columns, and Rename

In [ ]:
# Combine both cleaned datasets (LOS and NLOS)
df_clean_filtered = pd.concat([df_clean_filtered_LOS, df_clean_filtered_NLOS])

# Remove redundant 'NLOS' column if it exists
if "category" in df_clean_filtered.columns and "NLOS" in df_clean_filtered.columns:
    df_clean_filtered = df_clean_filtered.drop(columns=["NLOS"])

# Rename 'NLOS' column to 'Signal_Path' if it exists
df_clean_filtered = (
    df_clean_filtered.rename(columns={"NLOS": "Signal_Path"})
    if "NLOS" in df_clean_filtered.columns
    else df_clean_filtered
)

After removing the outliners, next we will be filtering the dataset based on **FP_IDX** values within a specified range and adds a new **Category** column that labels the data as **"LOS"** or **"NLOS"** based on the NLOS values.

In [ ]:
df_clean_filtered = df_clean[
    (df_clean["FP_IDX"] >= lower_bound) & (df_clean["FP_IDX"] <= upper_bound)
].copy()
# Create a new column to indicate the category (LOS or NLOS)
df_clean_filtered.loc[:, "Category"] = df_clean_filtered["NLOS"].apply(
    lambda x: "LOS" if x == 1.0 else "NLOS"
)

print(df_clean_filtered)

**Visualizing the Data**

The box plot below compares the distribution of **FP_IDX** values between **LOS** and **NLOS**. This helps in identifying differences in their statistical properties.

From the graph, we observe that **NLOS** has an outlier, indicating a potential anomaly, while **LOS** values are more evenly spread.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x="Category", y="FP_IDX", data=df_clean_filtered)
plt.title("Boxplot of FP_IDX (LOS vs NLOS)")
plt.show()

#### **1.1.2.3 - Target Variable: FP_AMP1, FP_AMP2**, **FP_AMP3**

**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

For FP_AMP1, 

In [ ]:
# Calculate the Range using IQR for FP_AMP1
Q1_FP_AMP1 = df_clean["FP_AMP1"].quantile(0.25)
Q3_FP_AMP1 = df_clean["FP_AMP1"].quantile(0.75)
IQR_FP_AMP1 = Q3_FP_AMP1 - Q1_FP_AMP1
lower_bound_FP_AMP1 = Q1_FP_AMP1 - 1.5 * IQR_FP_AMP1
upper_bound_FP_AMP1 = Q3_FP_AMP1 + 1.5 * IQR_FP_AMP1

# Identify outliers
outliers_FP_AMP1 = df_clean[
    (df_clean["FP_AMP1"] < lower_bound_FP_AMP1)
    | (df_clean["FP_AMP1"] > upper_bound_FP_AMP1)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP1 = df_clean[
    (df_clean["FP_AMP1"] >= lower_bound_FP_AMP1)
    & (df_clean["FP_AMP1"] <= upper_bound_FP_AMP1)
]

# Print outliers
print("Outliers for 'FP_AMP1':\n", outliers_FP_AMP1)

For FP_AMP2 ,

In [ ]:
# Calculate the Range using IQR for FP_AMP2
Q1_FP_AMP2 = df_clean["FP_AMP2"].quantile(0.25)
Q3_FP_AMP2 = df_clean["FP_AMP2"].quantile(0.75)
IQR_FP_AMP2 = Q3_FP_AMP2 - Q1_FP_AMP2
lower_bound_FP_AMP2 = Q1_FP_AMP2 - 1.5 * IQR_FP_AMP2
upper_bound_FP_AMP2 = Q1_FP_AMP2 + 1.5 * IQR_FP_AMP2

# Identify outliers
outliers_FP_AMP2 = df_clean[
    (df_clean["FP_AMP2"] < lower_bound_FP_AMP2)
    | (df_clean["FP_AMP2"] > upper_bound_FP_AMP2)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP2 = df_clean[
    (df_clean["FP_AMP2"] >= lower_bound_FP_AMP2)
    & (df_clean["FP_AMP2"] <= upper_bound_FP_AMP2)
]

# Print outliers
print("Outliers for 'FP_AMP2':\n", outliers_FP_AMP2)

For FP_AMP3 ,

In [ ]:
# Calculate the Range using IQR for FP_AMP3
Q1_FP_AMP3 = df_clean["FP_AMP3"].quantile(0.25)
Q3_FP_AMP3 = df_clean["FP_AMP3"].quantile(0.75)
IQR_FP_AMP3 = Q3_FP_AMP3 - Q1_FP_AMP3
lower_bound_FP_AMP3 = Q1_FP_AMP3 - 1.5 * IQR_FP_AMP3
upper_bound_FP_AMP3 = Q1_FP_AMP3 + 1.5 * IQR_FP_AMP3

# Identify outliers
outliers_FP_AMP3 = df_clean[
    (df_clean["FP_AMP3"] < lower_bound_FP_AMP3)
    | (df_clean["FP_AMP3"] > upper_bound_FP_AMP3)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP3 = df_clean[
    (df_clean["FP_AMP3"] >= lower_bound_FP_AMP3)
    & (df_clean["FP_AMP3"] <= upper_bound_FP_AMP3)
]

# Print outliers
print("Outliers for 'FP_IDX':\n", outliers_FP_AMP3)

After removing the outliers, we combine FP_AMP1, FP_AMP2, and FP_AMP3 into one DataFrame

In [ ]:
# Combining multiplte dataframe in to one DataFrame
df_filtered = pd.DataFrame(
    {
        "FP_AMP1": df_clean_filtered_FP_AMP1["FP_AMP1"],
        "FP_AMP2": df_clean_filtered_FP_AMP2["FP_AMP2"],
        "FP_AMP3": df_clean_filtered_FP_AMP3["FP_AMP3"],
    }
)

# Reshape the data for the violin plot
df_filtered_melted = df_filtered.melt(var_name="Feature", value_name="Value")

**Visualizing the Data**

After combining all into one DataFrame, we will plot a **violin graph** to illustrate the spread and median of **FP_AMP1**, **FP_AMP2**, and **FP_AMP3**, as their ranges are similar.

From the graph, we can see that these features have similar ranges, spread, and median. However, the median and spread for **FP_AMP1** are slightly higher than those of **FP_AMP2** and **FP_AMP3**, suggesting that **FP_AMP1** has slightly more variation or a higher central tendency compared to the other two.

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x="Feature", y="Value", data=df_filtered_melted)

plt.title("Distribution of FP_AMP1, FP_AMP2, and FP_AMP3 (Outliers Removed)")
plt.show()

#### **1.1.2.4 - Target Variable: STDEV_NOISE**


For **STDEV_NOISE**, we will plot a strip plot to gain a better understanding of the distribution. The strip plot below illustrates the distribution of **STDEV_NOISE** with labels 0.0 and 1.0.

From the graph, we can observe that the 0.0 label has a lower **STDEV_NOISE** value compared to 1.0, indicating that the noise is more consistent in **LOS** (Line-of-Sight) than in **NLOS** (Non-Line-of-Sight).

In [ ]:
sns.stripplot(data=df_clean, x="NLOS", y="STDEV_NOISE", jitter=True)
plt.title("Distribution of STDEV_NOISE for LOS and NLOS")
plt.xlabel("NLOS (LOS vs NLOS)")
plt.ylabel("STDEV_NOISE")
plt.show()

#### **1.1.2.5 - Target Variable: CIR_PWR**

**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

1. Calculate IQR and Identify Outliers

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) for CIR_PWR
Q1_CIR_PWR = df_clean["CIR_PWR"].quantile(0.25)
Q3_CIR_PWR = df_clean["CIR_PWR"].quantile(0.75)

# Calculate the Interquartile Range (IQR) for CIR_PWR
IQR_CIR_PWR = Q3_CIR_PWR - Q1_CIR_PWR

# Calculate lower and upper bounds for outliers
lower_bound_CIR_PWR = Q1_CIR_PWR - 1.5 * IQR_CIR_PWR
upper_bound_CIR_PWR = Q3_CIR_PWR + 1.5 * IQR_CIR_PWR

# Identify outliers in CIR_PWR based on the calculated bounds
outliers_CIR_PWR = df_clean[
    (df_clean["CIR_PWR"] < lower_bound_CIR_PWR)
    | (df_clean["CIR_PWR"] > upper_bound_CIR_PWR)
]

2. Filter and Clean Data

In [ ]:
# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_CIR_PWR = df_clean[
    (df_clean["CIR_PWR"] >= lower_bound_CIR_PWR)
    & (df_clean["CIR_PWR"] <= upper_bound_CIR_PWR)
]

# Print outliers
print("Outliers for 'CIR_PWR':\n", outliers_CIR_PWR)

**Visualizing the Data**

After removing the outliers, we will plot a histogram to better understand the distribution of **CIR_PWER** values. T

The histogram below illustrates the spread of values for **CIR_PWER** with labels 0.0 and 1.0. Based on this graph, we can see that the distributions for XLOS and NLOS statuses are similar for **CIR_PWER**, but **NLOS** has a wider spread of values.

In [ ]:
sns.histplot(data=df_clean_filtered_CIR_PWR, x="CIR_PWR", hue="NLOS", kde=True)
plt.title("CIR_PWR Distribution for LOS vs NLOS")
plt.xlabel("CIR_PWR")
plt.ylabel("Count")
plt.grid(True)
plt.show()

#### **1.1.2.6 - Target Variable: MAX_NOISE**

For **MAX_NOISE**, we will plot a violin plot to better understand the distribution and observe the variation in noise. The plot below illustrates the distribution of MAX_NOISE with labels 0.0 and 1.0.

From the graph, we can observe how the noise varies between LOS (Line-of-Sight) and NLOS (Non-Line-of-Sight), highlighting the differences in their noise patterns.

In [ ]:
df_clean_filtered = df_clean[
    (df_clean["MAX_NOISE"] >= 500) & (df_clean["MAX_NOISE"] < 1100)
]

sns.violinplot(data=df_clean_filtered, x="NLOS", y="MAX_NOISE")
plt.title("Distribution of MAX_NOISE for LOS and NLOS Classes")
plt.xlabel("NLOS (LOS vs NLOS)")
plt.ylabel("MAX_NOISE")
plt.grid(True)
plt.show()

#### **1.1.2.7 - Target Variable: RXPACC**

**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

Identify Outliers Using IQR for RXPACC

In [ ]:
# Filter the data based on specific RXPACC values (between 500 and 1100)
df_clean_filtered_RXPACC = df_clean[
    (df_clean["RXPACC"] >= 500) & (df_clean["RXPACC"] < 1100)
]

print(df_clean_filtered_RXPACC)

Next, we will remove columns that are not necessary for plotting the graph. This will help streamline the dataset, ensuring that we only work with the relevant features needed for visualization.

In [ ]:
# List of columns to exclude
columns_to_exclude = [
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "CIR_PWR",
    "MAX_NOISE",
]

# Remove the CIR_[Number] columns using regex
columns_to_exclude += [
    col for col in df_clean_filtered_RXPACC.columns if col.startswith("CIR")
]
df_clean_copy = df_clean_filtered_RXPACC.drop(columns=columns_to_exclude)


print(df_clean_copy)

**Visualizing the Data**

For **RXPACC** , we will plot a **KDE plot** to compare the distribution of **RXPACC** for **LOS (0.0)** and **NLOS (1.0)** classes. The plot helps highlight the density of **RXPACC** values, making it easy to spot any differences.

From the graph, we can see that **NLOS** has a more concentrated distribution of **RXPACC** values, while **LOS** shows a wider spread.

In [ ]:
sns.kdeplot(
    data=df_clean_copy, x="RXPACC", hue="NLOS", fill=True, palette={0: "blue", 1: "red"}
)
plt.title("KDE Plot of RXPACC for LOS (0.0) and NLOS (1.0)")
plt.xlabel("RXPACC")
plt.ylabel("Density")
plt.grid(True)
plt.show()

#### **1.1.2.8 - Target Variable: FRAME_LEN**

**Count Occurrences of Values**

In [ ]:
count_framelen = df_clean["FRAME_LEN"].value_counts()
print(count_framelen)

**Visualizing the Data**

For **FRAME_LEN**, we will plot a bar chart showing the count distribution, which allows us to see how the values are spread across the dataset.

From this graph, we can observe that **39.0** appears the most frequently, followed by **27.0**, with **29.0** being very rare in the dataset.`

In [ ]:
plt.figure(figsize=(10, 6))
count_framelen.plot.bar(color=["#3498db", "#e74c3c", "#f39c12"])
plt.title("Distribution of FRAME_LEN")
plt.ylabel("Count")
plt.xlabel("FRAME_LEN")
plt.xticks(rotation=0)

plt.yscale("linear")
plt.ylim(0, count_framelen.max() + 1000)

plt.show()

#### **1.1.2.9 - Target Variable: PREAM_LEN**

**Count Occurrences of Values**

In [ ]:
count_PREAM_LEN = df_clean["PREAM_LEN"].value_counts()
print(count_PREAM_LEN)

**Visualizing the Data**

For **PREAM_LEN**, we will plot a bar chart showing the count distribution, which allows us to see how the values are spread across the dataset.

From this graph, we can observe that **1024.0** is overwhelmingly more common than **1536.0**, with **1024.0** making up the vast majority of the dataset.

In [ ]:
plt.figure(figsize=(10, 6))
count_PREAM_LEN.plot.bar(color=["#3498db", "#e74c3c"])
plt.title("Distribution of PREAM_LEN")
plt.ylabel("Total Count")
plt.xticks(rotation=0)
plt.show()

#### **1.1.2.10 - Target Variable: PRFR and BITRATE**

**Count Occurrences of Values**

In [ ]:
# For 'PRFR' values
count_PRFR = df_clean["PRFR"].value_counts()

# For 'BITRATE' values
count_BITRATE = df_clean["BITRATE"].value_counts()

**Combine and Plot the Distribution of 'PRFR' and 'BITRATE'**

After counting the Occurense for PRFR and BITRATE, the graph will display side-by-side bars showing their total count occurrences. Since the columns have fixed categories, the graph will provide a clear comparison of how each category in PRFR and BITRATE is distributed.

In [ ]:
combined_df = pd.DataFrame(
    {"PRFR": count_PRFR, "BITRATE": count_BITRATE, "BITRATE": count_BITRATE}
).fillna(0) #combine

combined_df.plot(kind="bar", figsize=(10, 6), color=["#3498db", "#e74c3c"])

plt.title("Combined Distribution of PRFR and BITRATE")
plt.ylabel("Total Count")
plt.xlabel("Categories")
plt.xticks(rotation=0)  

plt.tight_layout()
plt.show()

#### **1.1.2.11 - Target Variable: RANGE**

**Identifying and Removing Outliers**

We be removing outliners Using IQR (Interquartile Range): 

In [ ]:
Q1_RANGE = df_clean["RANGE"].quantile(0.25)
Q3_RANGE = df_clean["RANGE"].quantile(0.75)
IQR_RANGE = Q3_RANGE - Q1_RANGE
lower_bound_RANGE = Q1_RANGE - 1.5 * IQR_RANGE
upper_bound_RANGE = Q3_RANGE + 1.5 * IQR_RANGE

# Identify outliers
outliers_RANGE = df_clean[
    (df_clean["RANGE"] < lower_bound_RANGE) | (df_clean["RANGE"] > upper_bound_RANGE)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_RANGE = df_clean[
    (df_clean["RANGE"] >= lower_bound_RANGE) & (df_clean["RANGE"] <= upper_bound_RANGE)
]

print(df_clean_filtered_RANGE)

**Visualizing the Data**

For **RANGE**, we will plot a box plot to understand its distribution and identify outliers. 

From the graph, we can observe that **RANGE** has a wide spread with some potential outliers at the higher end.

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot(
    df_clean_filtered_RANGE["RANGE"],
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="darkblue"),
    medianprops=dict(color="red"),
)
plt.title("Box Plot of RANGE")
plt.xlabel("RANGE")
plt.show()

### **1.1.3 - Feature Evaluation and Relationship Analysis**

After having a better understanding of the features in the dataset using visualization, next we need to identify which features have the most impact on predicting the target variable. This allows us to focus on the most relevant features, improving model performance and interpretability.

We be using different technique:


##### **1. Correlation Matrix**

Firstly we be calculating the  **Correlation Matrix**, we will plot a heatmap to gain a better understanding of the relationships between different features. The heatmap below illustrates the correlation between each feature, with color intensity representing the strength of the relationship.

From the graph, we can observe that **RXPACC** has a strong positive correlation with **RANGE**, while **FP_AMP1**, **FP_AMP2**, and **FP_AMP3** show negative correlations with **RANGE**, indicating their potential influence on the target variable.

In [ ]:
features = [
    "RANGE",
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "STDEV_NOISE",
    "CIR_PWR",
    "MAX_NOISE",
    "RXPACC",
    "FRAME_LEN",
    "PREAM_LEN",
]

correlation_matrix = df_clean[features].corr()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

##### **2. Mutual Information**

Secondly, we will calculate the **Mutual Information** to assess the relationship between each feature and the target variable **RANGE**. We will plot a bar chart to visualize the mutual information scores where higher scores indicate more significant features for prediction.

From the graph, we can observe that **RXPACC** has the highest mutual information score  making it the most important feature for predicting **RANGE**. Other features such as **MAX_NOISE** and **CIR_PWR** also have notable scores  suggesting their relevance. On the other hand, **FRAME_LEN** has a low score which indicating that it might have less influence on the target variable.

In [ ]:
# import libraries
try:
    from sklearn.feature_selection import mutual_info_regression

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
# mutual information => (to check the relationship between the features)

# Features
features = [
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "STDEV_NOISE",
    "CIR_PWR",
    "MAX_NOISE",
    "RXPACC",
    "FRAME_LEN",
    "PREAM_LEN",
]

# Define X as features set
X = df_clean[features]

# Define target variable
y = df_clean["RANGE"]

# Compute mutual information scores
mi_scores = mutual_info_regression(X, y)
mi_scores = pd.Series(mi_scores, index=features)

# Sort the scores in descending order and print
mi_scores = mi_scores.sort_values(ascending=False)
print("Mutual Information Scores (in descending order):")
print(mi_scores)

**Visualizing the Data**

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=mi_scores.index, y=mi_scores.values)
plt.xlabel("Features")
plt.ylabel("Mutual Information Score")
plt.title("Feature Importance using Mutual Information")
plt.xticks(rotation=45)
plt.show()

##### **3. Normalization**

Thirdly, we performed **Normalization** using **MinMaxScaler** to scale feature values between **0 and 1**. The histogram below shows the distribution of scaled features, ensuring they fall within the same range.

Most features are **well-scaled**, but some may show **skewness** or **concentration** at one end, indicating the need for further transformations. The histogram also highlights potential **outliers** that could impact model performance.

Normalization helps ensure **equal contribution** of all features, improving the **accuracy** and **stability** of the classifier.

In [ ]:
# import libraries

try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
X = df_clean.drop(columns=["NLOS"])

y = df_clean["NLOS"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)

print(X_train)

##### **4. Principal Component Analysis (PCA)**

Lastly, we performed **PCA** to reduce the number of CIR features while retaining key information. PCA transforms the original features into principal components, which capture the most important patterns in the data. The first 14 components explain 95% of the variance, with the 15th capturing the remaining variance. This reduces the dataset's complexity while preserving essential details for further analysis and modeling.

In [ ]:
# Import  libraries
try:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

1. Selecting CIR Features from the Cleaned Data


In [ ]:
# CIR Features List
cir_features = ["NLOS"]

X_cir = df_clean[cir_features]  # Extract relevant features from the DataFrame

2. Standardization of Data

In [ ]:
scaler = StandardScaler()
X_cir_scaled = scaler.fit_transform(X_cir)

3. Applying PCA

- To retain 95% of the variance by selecting the number of components.


In [ ]:
pca = PCA(n_components=0.95)  # Retains enough components => 95% of the variance
X_cir_pca = pca.fit_transform(X_cir_scaled)

4. Checking the Shape of Transformed Data and Converting PCA Data to DataFrame

In [ ]:
reduced_features = X_cir_pca.shape[1]
print(
    f"New shape after PCA: {X_cir_pca.shape} (Reduced to {reduced_features} components)"
)

pca_feature_names = [f"PCA_{i}" for i in range(reduced_features)]
df_pca = pd.DataFrame(X_cir_pca, columns=pca_feature_names)

# df_clean = df_clean.drop(columns=cir_features).reset_index(drop=True)
# df_clean = pd.concat([df_clean, df_pca], axis=1)

5. Replacing Original Features with PCA Features

In [ ]:
explained_variance_ratio = pca.explained_variance_ratio_  # value
cumulative_variance = np.cumsum(explained_variance_ratio)  # ratio

print("Explained Variance Ratio for each component:", explained_variance_ratio)
print("Cumulative Explained Variance:", cumulative_variance)

**Visualizing the Data**

In [ ]:
plt.figure(figsize=(8, 5))
df_pca["PCA_0"].value_counts().plot(
    kind="bar", alpha=0.7, color="skyblue", edgecolor="black"
)
plt.title("Bar Plot of PCA_0")
plt.xlabel("PCA_0 Value")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(True)
plt.show()

### **1.1.4 - Data Mining & Splitting**
After feature evaluation, we divide the dataset into training and test sets to train the model and assess its performance on unseen data.

In [ ]:
# import libaries
try:
    from sklearn.model_selection import train_test_split
    import numpy as np

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

First, we split the dataset into training and testing sets to maintains the same structure but improves readability.

In [ ]:
# Remove CIR columns => PCA already reduced them
df_clean = df_clean.loc[:, ~df_clean.columns.str.match(r"^CIR\d+$")]
df_clean = df_clean.drop(columns=["FP_AMP2", "MAX_NOISE", "FRAME_LEN"])


# Combine FP_AMP columns => into one average (only if they exist)
fp_amp_cols = df_clean.filter(like="FP_AMP").columns  # Find all FP_AMP columns
if len(fp_amp_cols) > 0:
    df_clean["FP_AMP_AVG"] = df_clean[fp_amp_cols].mean(axis=1)
    df_clean = df_clean.drop(columns=fp_amp_cols, errors="ignore")

# Define features and target
X = df_clean.drop(columns=["NLOS"], errors="ignore")
y = df_clean["NLOS"]

# Split data into train and test 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print shape of the datasets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

#### **1.2 - Model Training**

Training the model with data and tuning its settings to improve accuracy on new data.

In [ ]:
# import libaries
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        roc_curve,
        roc_auc_score,
        accuracy_score,
        confusion_matrix,
        classification_report,
    )

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

##### **1.2.1 - Algorithm 1 (Training & Model Evaluation)**

In [ ]:
# import libaries
try:
    from sklearn.metrics import auc

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

**Logistic Regression** 

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)  # train

**Model Evaluation**

In [ ]:
y_pred_lr = lr_model.predict(X_test)  # Prediction
accuracy_lr = accuracy_score(y_test, y_pred_lr)  # Accuracy
conf_matrix_lr = confusion_matrix(y_test, y_pred_lr)# Confusion Matrix

comparison_lr = pd.DataFrame(
    {"Actual": y_test[:10].values, "Predicted": y_pred_lr[:10]}
)

print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")
print(f"Logistic Regression Confusion Matrix:\n{conf_matrix_lr}")
print(
    f"Logistic Regression Classification Report:\n{classification_report(y_test, y_pred_lr)}"
)
print("\nFirst 10 Predicted vs. Actual Values (Logistic Regression):\n", comparison_lr)

**Training and Test ROC-AUC** 

In [ ]:
# Training
y_train_pred_lr = lr_model.predict_proba(X_train)[:, 1]
train_auc_lr = roc_auc_score(y_train, y_train_pred_lr)

# Test AUC
y_test_pred_lr = lr_model.predict_proba(X_test)[:, 1]
test_auc_lr = roc_auc_score(y_test, y_test_pred_lr)

print(
    f"Logistic Regression - Train ROC-AUC: {train_auc_lr:.4f}, Test ROC-AUC: {test_auc_lr:.4f}"
)

**Visualizing the Data**

**ROC-AUC Training**

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    conf_matrix_lr,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"],
)
plt.title("Confusion Matrix (Logistic Regression)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_test_pred_lr)
roc_auc_lr = auc(fpr_lr, tpr_lr)

# Plot ROC curve
plt.figure(figsize=(6, 6))
plt.plot(
    fpr_lr, tpr_lr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc_lr:.4f})"
)
plt.plot(
    [0, 1], [0, 1], color="gray", linestyle="--"
)  # Diagonal line (no discrimination)
plt.title("ROC Curve (Logistic Regression)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

##### **1.2.2 - Algorithm 2 (Training & Model Evaluation)**

In [ ]:
# import libaries
try:
    from sklearn.svm import SVC

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

**Support Vector Machine (SVM) (WIP)** 

In [ ]:
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train, y_train)  # train

**Model Evaluation**

In [ ]:
y_pred_svm = svm_model.predict(X_test)  # prediction
accuracy_svm = accuracy_score(y_test, y_pred_svm)  # Accuracy
conf_matrix_svm = confusion_matrix(y_test, y_pred_svm)  # Confusion Matrix
comparison_svm = pd.DataFrame(
    {"Actual": y_test[:10].values, "Predicted": y_pred_svm[:10]}
)  # First 10 for Predicted vs Actual Values

print(f"SVM Accuracy: {accuracy_svm:.4f}")
print(f"SVM Confusion Matrix:\n{conf_matrix_svm}")
print(f"SVM Classification Report:\n{classification_report(y_test, y_pred_svm)}")
print("\nFirst 10 Predicted vs. Actual Values (SVM):\n", comparison_svm)

**Training and Test ROC-AUC** 

In [ ]:
# Training
y_train_pred_svm = svm_model.predict_proba(X_train)[:, 1]
train_auc_svm = roc_auc_score(y_train, y_train_pred_svm)

# Test
y_test_pred_svm = svm_model.predict_proba(X_test)[:, 1]
test_auc_svm = roc_auc_score(y_test, y_test_pred_svm)

print(f"SVM - Train ROC-AUC: {train_auc_svm:.4f}, Test ROC-AUC: {test_auc_svm:.4f}")

**Visualizing the Data**

**ROC-AUC Training**

In [ ]:
#  ROC Curve for SVM (Training Data)
fpr_train_svm, tpr_train_svm, _ = roc_curve(y_train, y_train_pred_svm)
roc_auc_train_svm = auc(fpr_train_svm, tpr_train_svm)

# ROC Curve for SVM (Test Data)
fpr_test_svm, tpr_test_svm, _ = roc_curve(y_test, y_test_pred_svm)
roc_auc_test_svm = auc(fpr_test_svm, tpr_test_svm)

# Plot ROC Curves
plt.figure(figsize=(10, 8))

#  Training
plt.plot(
    fpr_train_svm,
    tpr_train_svm,
    color="blue",
    label=f"Train ROC Curve (AUC = {roc_auc_train_svm:.2f})",
)

# Test
plt.plot(
    fpr_test_svm,
    tpr_test_svm,
    color="green",
    label=f"Test ROC Curve (AUC = {roc_auc_test_svm:.2f})",
)

# Plot the diagonal =>(random classifier)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM ROC Curve: Train vs Test")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

##### **1.2.3 - Algorithm 3 (Training & Model Evaluation)**

**Random Forest**

In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Hyperparameter grid for tuning
param_grid = {
    "n_estimators": [100, 200, 300],  # Number of trees in the forest
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 5, 10],  # Split a node
    "min_samples_leaf": [1, 2, 4],  # Each leaf node
}

# tune hyperparameters
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

random_search.fit(X_train, y_train)  
best_rf_model = random_search.best_estimator_
print("Best Hyperparameters:", random_search.best_params_)

**Model Evaluation**

In [ ]:
y_pred_rf = best_rf_model.predict(X_test)  # predict
accuracy_rf = accuracy_score(y_test, y_pred_rf)  # Accuracy
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)  # Confusion Matrix
comparison_rf = pd.DataFrame(
    {"Actual": y_test[:10].values, "Predicted": y_pred_rf[:10]}
)  # First 10 for Predicted vs Actual Values

print(f"Random Forest Accuracy: {accuracy_rf:.4f}")
print(f"Random Forest Confusion Matrix:\n{conf_matrix_rf}")
print(
    f"Random Forest Classification Report:\n{classification_report(y_test, y_pred_rf)}"
)
print("\nFirst 10 Predicted vs. Actual Values (Random Forest):\n", comparison_rf)

**Training and Test ROC-AUC** 

In [ ]:
# Training
y_train_pred_rf = best_rf_model.predict_proba(X_train)[:, 1]
train_auc_rf = roc_auc_score(y_train, y_train_pred_rf)

# Test
y_test_pred_rf = best_rf_model.predict_proba(X_test)[:, 1]
test_auc_rf = roc_auc_score(y_test, y_test_pred_rf)

print(
    f"Random Forest - Train ROC-AUC: {train_auc_rf:.4f}, Test ROC-AUC: {test_auc_rf:.4f}"
)

**Visualizing the Data**

**ROC-AUC Training**

In [ ]:
# For Training Data
fpr_train_rf, tpr_train_rf, _ = roc_curve(y_train, y_train_pred_rf)
roc_auc_train_rf = auc(fpr_train_rf, tpr_train_rf)

# For Training Data
fpr_test_rf, tpr_test_rf, _ = roc_curve(y_test, y_test_pred_rf)
roc_auc_test_rf = auc(fpr_test_rf, tpr_test_rf)

plt.figure(figsize=(10, 8))

# Training 
plt.plot(
    fpr_train_rf,
    tpr_train_rf,
    color="blue",
    label=f"Train ROC Curve (AUC = {roc_auc_train_rf:.2f})",
)

#  Test
plt.plot(
    fpr_test_rf,
    tpr_test_rf,
    color="green",
    label=f"Test ROC Curve (AUC = {roc_auc_test_rf:.2f})",
)

# Plot the diagonal => (random classifier)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest ROC Curve: Train vs Test")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

#### **1.3 - Model Comparison and Analysis**

##### **1.3.1 - Model Evaluation**

After identifying important features, we use the model to predict the test data and evaluate its performance. The classification report provides key metrics like precision, recall, and F1-score

##### **1.3.2 - Model Evaluation on Test Data**

Now let's evaluate the model on the test dataset to see how it performs on unseen data.

In [ ]:
# Predict on the test set
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", classification_report(y_test, y_pred))

**Print first 10 predicted vs. actual values**

In [ ]:
comparison_df = pd.DataFrame({"Actual": y_test[:10].values, "Predicted": y_pred[:10]})
print("\nFirst 10 Predicted vs. Actual Values:\n", comparison_df)

##### **1.3.3 - ROC Curve Analysis**

In [ ]:
fpr, tpr, _ = roc_curve(y_test, best_rf_model.predict_proba(X_test)[:, 1])

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="b", label="ROC Curve")
plt.plot([0, 1], [0, 1], color="r", linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test Set ROC Curve")
plt.legend(loc="lower right")
plt.show()

#### **1.4 - Model Deployment**
After selecting and evaluating the best model, we save the trained model for future use and easy 

In [ ]:
# import libraries

try:
    import os
    import joblib
    from sklearn.metrics import accuracy_score, roc_auc_score

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
# Save the final trained model
folder_name = "final_models"
os.makedirs(folder_name, exist_ok=True)

# # final model
# joblib.dump(best_rf_model, folder_name+"/Final_trained_model.pkl")  # Save model
# loaded_model = joblib.load(folder_name+"/Final_trained_model.pkl")

# print("Final trained model saved as 'final_model/trained_model.pkl'")

# Logistic Regression Model
LR_filename = folder_name + "/logistic_regression_model.pkl"
joblib.dump(lr_model, LR_filename)
print(f"Logistic Regression model saved as ${LR_filename}")

# SVM Model
SVM_name = folder_name + "/SVM_model.pkl"
joblib.dump(lr_model, SVM_name)
print(f"SVM model saved as ${SVM_name}")

# Random Forest Model
RM_filename = folder_name + "/RF_model.pkl"
joblib.dump(best_rf_model, RM_filename)
print(f"Random Forest model saved as ${RM_filename}")